# 01 · Deep extreme quantile regression and deep GPD regression for US wildfires

> **Source.** The code in this notebook comes from the CASE 2025 short course *AI and extremes* by **Likun Zhang, Jordan Richards and Reetam Majumder** ([Jbrich95/CASE2025_shortcourse](https://github.com/Jbrich95/CASE2025_shortcourse)). It is converted from the original `.Rmd` to a Google Colab notebook, with explanatory notes added. Changes to the original code are marked **[Adapted]** and explained where they occur.

**How to run on Google Colab.** Open this notebook in Colab, choose *Runtime → Change runtime type → R* (CPU is enough), then *Runtime → Run all*. The setup cell installs the R packages and downloads the data from the course repository. The first time Keras is used, R may spend a few minutes setting up Python/TensorFlow.

**What this notebook covers**
1. A standard neural-network regression, with checkpointing to control overfitting.
2. Deep **quantile regression** with the pinball (tilted) loss, to estimate a covariate-dependent threshold $u(\mathbf x)$.
3. Deep **peaks-over-threshold** modelling: exceedances above $u(\mathbf x)$ follow a generalised Pareto distribution (GPD) with scale $\sigma(\mathbf x)$ and shape $\xi(\mathbf x)$.
4. Three increasingly parsimonious models for $\xi$, compared by test loss.
5. Post-hoc interpretation of the fitted scale with **LIME**.

The method follows Richards & Huser, *Extreme Quantile Regression with Deep Learning*, Chapter 21 of the *Handbook on Statistics of Extremes*. The data are from Richards & Huser (2022, arXiv:2208.07581).

In [ ]:
# [Adapted] Setup that works on Google Colab's R runtime and on a local machine.
options(timeout = 1200)                       # large data file on slow connections
on_colab <- dir.exists("/content")
if (on_colab) install.packages(c("keras3", "evd", "lime"))
# Local machines: use a Python virtualenv called "r-keras" if one exists
if (!on_colab && reticulate::virtualenv_exists("r-keras")) {
  reticulate::use_virtualenv("r-keras", required = TRUE)
}
library(keras3)

# Course data, downloaded from the original repository if not already present
base_url <- "https://github.com/Jbrich95/CASE2025_shortcourse/raw/main/"
for (f in c("USWild.rda")) {
  if (!file.exists(f)) download.file(paste0(base_url, f), f, mode = "wb")
}
dir.create("checkpoints", showWarnings = FALSE)   # [Adapted] folder used by the checkpoint callbacks

# If the first Keras call fails with a Python/TensorFlow error, run
#   keras3::install_keras()
# then restart the session (Runtime -> Restart session) and run this cell again.

Set seeds for reproducibility. Results can still differ slightly between machines and TensorFlow versions.

In [ ]:
set.seed(1)
tensorflow::set_random_seed(1)

## 1 · The data

Monthly US burnt area from wildfires on a 129 × 61 grid, with 22 covariates. Only the last five years are used: rows 127–161, which are **2011–2015, March to September**, giving 7 months × 5 years = 35 months.

In [ ]:
load("USWild.rda")

Y = USWild$BA[127:161, , ]

X = array(dim = c(35, 129, 61, 22))
X[, , , 1] = USWild$X$X.t2m[127:161, , , ]
X[, , , 2] = USWild$X$X.SPI[127:161, , , ]
X[, , , -c(1, 2)] = USWild$X$X.N[127:161, , , 1:20]

print(matrix(USWild$times[127:161], nrow = 7))   # [Adapted] shows which months are used

The first two covariates are monthly 2 m air temperature (`t2m`) and the standardised precipitation index (`SPI`). The rest are meteorological and land-cover descriptors.

In [ ]:
dim(X)
cov.names <- c("t2m", "SPI", USWild$X$cov.names[1:20])
print(cov.names)

The response is monthly burnt area. Burnt area is *very* heavy-tailed, so the models use its square root for numerical stability. Grid cells outside the study region are coded with large negative values. Taking logs of those cells in the map plots produces harmless `NaNs produced` warnings: the cells are simply left blank.

In [ ]:
image(log(1 + Y[1, , ]), col = heat.colors(20, rev = T), asp = 0.625)

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 5)   # [Adapted] wider plots in Jupyter/Colab
par(mfrow = c(1, 2))
image(X[1, , , 1], col = heat.colors(20, rev = T), main = "temperature", asp = 0.625)
image(X[1, , , 2], col = terrain.colors(20, rev = F), main = "SPI", asp = 0.625)

Keep only cells with positive burnt area, flattened into vectors, and take the square root of the response.

In [ ]:
X.positive <- apply(X, 4, function(x) x[Y > 0])
Y.positive <- sqrt(Y[Y > 0])
length(Y.positive)

Standardise covariates to zero mean and unit variance. This makes training more stable.

In [ ]:
means <- apply(X.positive, 2, mean)
sds <- apply(X.positive, 2, sd)
X.scaled <- apply(as.matrix(1:ncol(X.positive)), 1, function(ind)
  (X.positive[, ind] - means[ind]) / sds[ind])

> **Validation note.** The means and standard deviations above are computed from *all* rows, including those that later become test data. That is mild information leakage. Notebook **04** standardises using training rows only.

## 2 · Standard prediction

Target $E[\log Y \mid \mathbf x]$ with the mean-squared-error loss. Sample 45,000 rows for training and 5,000 for testing.

In [ ]:
n = length(Y.positive)
train_sample_idx <- sample(1:n, 45000)
test_sample_idx <- sample((1:n)[-train_sample_idx], 5000)

Y_train <- Y.positive[train_sample_idx]
X_train <- X.scaled[train_sample_idx, ]
Y_test <- Y.positive[test_sample_idx]
X_test <- X.scaled[test_sample_idx, ]

> **Validation note.** Rows are individual grid-cell-months sampled at random. Neighbouring cells in the same month are strongly dependent, so many test rows have a near-identical "twin" in the training data, and test performance is optimistic about prediction for an unseen period. Notebook **04** measures how large this effect is.

A multilayer perceptron (MLP): two hidden layers of width 64 with ReLU activations, and a linear output.

In [ ]:
model <- keras_model_sequential()
model %>%
  layer_dense(units = 64, activation = 'relu') %>%
  layer_dense(units = 64, activation = 'relu') %>%
  layer_dense(units = 1, activation = 'linear')

model %>% compile(
  optimizer = "adam",
  loss = "mean_squared_error",
  metrics = list("mean_absolute_error")
)

Train for 100 epochs. Keras holds out the last 20% of the training rows as validation data (`validation_split = 0.2`). Gradients are never computed on the validation rows; they are only used to monitor generalisation.

In [ ]:
history <- model %>% fit(
  x = as.matrix(X_train),
  y = as.matrix(log(Y_train)),
  epochs = 100,
  validation_split = 0.2,
  verbose = 0,
  shuffle = T
)
plot(history)
summary(model)

The validation loss stops improving while the training loss keeps falling: the model is **overfitting**. The fix is to **checkpoint**, saving the weights from the epoch with the lowest validation loss and reloading them at the end.

In [ ]:
model <- keras_model_sequential() %>%
  layer_dense(units = 64, activation = 'relu') %>%
  layer_dense(units = 64, activation = 'relu') %>%
  layer_dense(units = 1, activation = 'linear') %>%
  compile(optimizer = "adam", loss = "mean_squared_error",
          metrics = list("mean_absolute_error"))

checkpoint <- callback_model_checkpoint(
  paste0("checkpoints/LSE.weights.h5"),
  monitor = "val_loss", verbose = 0,
  save_best_only = TRUE, save_weights_only = TRUE,
  mode = "min", save_freq = "epoch"
)

history <- model %>% fit(x = X_train, y = as.matrix(log(Y_train)),
                         epochs = 50, batch_size = 16, verbose = 0,
                         callbacks = list(checkpoint),
                         validation_split = 0.2)
plot(history)

model <- load_model_weights(model, filepath = paste0("checkpoints/LSE.weights.h5"))

In [ ]:
predictions <- model %>% predict(X_test)
plot(log(Y_test), predictions)
abline(a = 0, b = 1)

For comparison, fit a linear model. A linear model is just a one-layer network with a single neuron and a linear activation.

In [ ]:
linear.model <- keras_model_sequential() %>%
  layer_dense(units = 1, activation = 'linear') %>%
  compile(optimizer = "adam", loss = "mean_squared_error",
          metrics = list("mean_absolute_error"))

history <- linear.model %>% fit(x = X_train, y = as.matrix(log(Y_train)),
                                epochs = 15, verbose = 0, validation_split = 0.2)
plot(history)

linear_predictions <- linear.model %>% predict(X_test)
plot(log(Y_test), linear_predictions, asp = 1)
abline(a = 0, b = 1)

Map the predictions for one month. Predictions use the full covariate grid, so it must be scaled with the same means and standard deviations as the training data.

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 5)
par(mfrow = c(1, 3))
t <- 1
X.plot <- X[t, , , ]
dim(X.plot) <- c(prod(dim(X.plot)[1:2]), dim(X.plot)[3])
X.plot.scaled <- apply(as.matrix(1:ncol(X.plot)), 1, function(ind)
  (X.plot[, ind] - means[ind]) / sds[ind])

linear.predictions <- linear.model %>% predict(X.plot.scaled)
dim(linear.predictions) <- dim(Y)[2:3]
linear.predictions[Y[t, , ] < 0] = NA
image(log(Y[t, , ]), main = "Observation", asp = 0.625)
image(linear.predictions, main = "Linear model predictions", asp = 0.625)

NN.predictions <- model %>% predict(X.plot.scaled)
dim(NN.predictions) <- dim(Y)[2:3]
NN.predictions[Y[t, , ] < 0] = NA
image(NN.predictions, main = "MLP model predictions", asp = 0.625)

## 3 · Deep quantile regression

To estimate the $\tau$-quantile of $Y \mid \mathbf x$, minimise the **pinball (tilted) loss**

$$\ell_\tau(y, q) = \max\{\tau (y-q),\ (\tau-1)(y-q)\}.$$

Its expected value is minimised when $q$ is the true conditional $\tau$-quantile. Custom Keras losses take `(y_true, y_pred)` and must be differentiable, so they are written with Keras `op_*` functions.

*Note:* the original text mentions the 80% quantile, but the code sets $\tau = 0.85$. Everything below uses $\tau = 0.85$.

In [ ]:
tau <- 0.85
tilted_loss <- function(y_true, y_pred) {
  error = y_true - y_pred
  return(op_mean(op_maximum(tau * error, (tau - 1) * error)))
}

The final layer uses an **exponential** activation, so the predicted quantile is strictly positive (all responses here are positive).

In [ ]:
u.model <- keras_model_sequential() %>%
  layer_dense(units = 64, activation = 'relu') %>%
  layer_dense(units = 64, activation = 'relu') %>%
  layer_dense(units = 1, activation = "exponential")

u.model %>% compile(optimizer = "adam", loss = tilted_loss)

checkpoint <- callback_model_checkpoint(
  paste0("checkpoints/u.weights.h5"),
  monitor = "val_loss", verbose = 0,
  save_best_only = TRUE, save_weights_only = TRUE,
  mode = "min", save_freq = "epoch"
)

history <- u.model %>% fit(x = X_train, y = as.matrix(Y_train),
                           batch_size = 64, epochs = 60, verbose = 0,
                           callbacks = list(checkpoint), validation_split = 0.2)
plot(history)
u.model <- load_model_weights(u.model, filepath = paste0("checkpoints/u.weights.h5"))

Sanity check: the fraction of test responses *below* the predicted quantile should be close to $\tau$.

In [ ]:
test.pred.theshold <- u.model %>% predict(X_test)
mean(Y_test < test.pred.theshold)   # should be close to tau

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 5)
par(mfrow = c(1, 2))
t <- 3
X.plot <- X[t, , , ]
dim(X.plot) <- c(prod(dim(X.plot)[1:2]), dim(X.plot)[3])
X.plot.scaled <- apply(as.matrix(1:ncol(X.plot)), 1, function(ind)
  (X.plot[, ind] - means[ind]) / sds[ind])

image(log(Y[t, , ]), main = "Observation", asp = 0.625)
NN.predictions <- u.model %>% predict(X.plot.scaled)
dim(NN.predictions) <- dim(Y)[2:3]
NN.predictions[Y[t, , ] < 0] = NA
image(log(NN.predictions), main = "Threshold prediction", asp = 0.625)

## 4 · Deep peaks-over-threshold (GPD) regression

The **Pickands–Balkema–de Haan theorem** says that, for a high enough threshold, excesses $Y - u(\mathbf x)$ given $Y > u(\mathbf x)$ are approximately GPD$(\sigma(\mathbf x), \xi(\mathbf x))$. Compute the excesses above the estimated threshold, using the same train/test split.

In [ ]:
train.pred.threshold <- u.model %>% predict(X_train)
train.exceed_idx = which(Y_train > train.pred.threshold)
Y_train.excess = (Y_train - train.pred.threshold)[train.exceed_idx]
X_train.excess = X_train[train.exceed_idx, ]

test.exceed_idx = which(Y_test > test.pred.theshold)
Y_test.excess = (Y_test - test.pred.theshold)[test.exceed_idx]
X_test.excess = X_test[test.exceed_idx, ]

length(Y_train.excess)
length(Y_test.excess)

There are far fewer excesses than rows, so this network is smaller and uses sigmoid activations. The output layer has **two** units, $\sigma$ and $\xi$, both passed through an exponential to keep them positive.

> **Modelling note.** The exponential activation forces $\xi > 0$: the architecture *assumes* a heavy (Fréchet-type) tail rather than learning the sign of $\xi$. For burnt area that is a reasonable prior belief, but it is an assumption, not a result.

In [ ]:
GPD.model <- keras_model_sequential() %>%
  layer_dense(units = 32, activation = 'sigmoid') %>%
  layer_dense(units = 32, activation = 'sigmoid') %>%
  layer_dense(units = 2, activation = "exponential")

The loss is the GPD negative log-likelihood. For an excess $y > 0$,

$$-\log f(y) = \log \sigma + \left(\tfrac{1}{\xi} + 1\right) \log\left(1 + \xi\,\tfrac{y}{\sigma}\right).$$

Note that this is a **sum** over observations, not a mean. It therefore scales with sample size, which matters when comparing losses across datasets of different sizes.

In [ ]:
GPD_nll <- function(y_true, y_pred) {
  sigma = y_pred[all_dims(), 1]
  xi = y_pred[all_dims(), 2]
  y = y_true[all_dims(), 1]
  ll1 = -(1 / xi + 1) * op_log(1 + xi * y / sigma)
  ll2 = -op_log(sigma)
  return(-(op_sum(ll1 + ll2)))
}

Train with checkpointing plus **early stopping**: training stops if the validation loss has not improved for 10 epochs.

In [ ]:
GPD.model %>% compile(optimizer = "adam", loss = GPD_nll)

checkpoint <- callback_model_checkpoint(
  paste0("checkpoints/GPD.weights.h5"),
  monitor = "val_loss", verbose = 0,
  save_best_only = TRUE, save_weights_only = TRUE,
  mode = "min", save_freq = "epoch"
)
early.stopping <- callback_early_stopping(monitor = "val_loss", patience = 10)

history <- GPD.model %>% fit(x = X_train.excess, y = as.matrix(Y_train.excess),
                             batch_size = 16, epochs = 100, verbose = 0,
                             callbacks = list(checkpoint, early.stopping),
                             validation_split = 0.2)
plot(history)
GPD.model <- GPD.model %>% load_model_weights(filepath = paste0("checkpoints/GPD.weights.h5"))

In [ ]:
test_loss_GPD1 <- GPD.model %>% evaluate(X_test.excess, as.matrix(Y_test.excess),
                                         batch_size = length(Y_test.excess))
print(paste0("test loss for GPD NN: ", test_loss_GPD1))

test.GPD.estimates <- GPD.model %>% predict(X_test.excess)
test.sigma = test.GPD.estimates[, 1]
test.xi = test.GPD.estimates[, 2]
par(mfrow = c(1, 2))
hist(test.sigma, main = "sigma estimates")
hist(test.xi, main = "xi estimates")

**Goodness of fit with a pooled QQ plot.** Each test excess has its own $(\sigma_i, \xi_i)$, so their raw quantiles cannot be compared directly. Instead, apply the probability integral transform $F_{\text{GPD}}(y_i; \sigma_i, \xi_i)$, then map to unit-exponential margins with $-\log(1-\cdot)$. If the model is right, the transformed values are i.i.d. Exp(1), so one QQ plot covers all of them.

In [ ]:
exp.data = apply(cbind(Y_test.excess, test.sigma, test.xi), 1, function(x) {
  qexp(evd::pgpd(x[1], loc = 0, scale = x[2], shape = x[3]))
})
n_p = length(exp.data)
ps = (1:n_p) / (n_p + 1)
qs = quantile(exp.data, ps, na.rm = T)
par(mfrow = c(1, 1))
plot(qexp(ps), qs, ylab = "Fitted", xlab = "Theoretical",
     main = "MLP model 1", pch = 20, asp = 1)
abline(a = 0, b = 1, col = "red")

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 5)
par(mfrow = c(1, 3))
t <- 3
X.plot <- X[t, , , ]
dim(X.plot) <- c(prod(dim(X.plot)[1:2]), dim(X.plot)[3])
X.plot.scaled <- apply(as.matrix(1:ncol(X.plot)), 1, function(ind)
  (X.plot[, ind] - means[ind]) / sds[ind])

image(log(Y[t, , ]), main = "Observation", asp = 0.625)
NN.predictions <- GPD.model %>% predict(X.plot.scaled)
sigma.predictions <- NN.predictions[, 1]
dim(sigma.predictions) <- dim(Y)[2:3]
sigma.predictions[Y[t, , ] < 0] = NA
image(sigma.predictions, main = "sigma prediction", asp = 0.625)

xi.predictions <- NN.predictions[, 2]
dim(xi.predictions) <- dim(Y)[2:3]
xi.predictions[Y[t, , ] < 0] = NA
image(xi.predictions, main = "xi prediction", asp = 0.625)

## 5 · More parsimonious models for $\xi$

The shape parameter $\xi$ controls tail heaviness and is notoriously hard to estimate. Two simpler alternatives:

* **Model 2:** $\xi(\mathbf x)$ depends on temperature only and is bounded in $(0, 1)$ by a sigmoid output.
* **Model 3:** $\xi$ is constant.

These need the **functional** Keras API, with separate input branches for $\sigma$ and $\xi$ that are concatenated at the output.

In [ ]:
input_sigma <- layer_input(shape = ncol(X_train.excess), name = 'input_sigma')
input_xi <- layer_input(shape = 1, name = 'input_xi')

sigma_model <- input_sigma %>%
  layer_dense(units = 32, activation = 'sigmoid') %>%
  layer_dense(units = 32, activation = 'sigmoid') %>%
  layer_dense(units = 1, activation = 'exponential')

xi_model <- input_xi %>%
  layer_dense(units = 16, activation = 'relu') %>%
  layer_dense(units = 16, activation = 'relu') %>%
  layer_dense(units = 1, activation = 'sigmoid')

output <- layer_concatenate(c(sigma_model, xi_model))
GPD.model2 <- keras_model(inputs = c(input_sigma, input_xi), outputs = output)
GPD.model2 %>% compile(optimizer = "adam", loss = GPD_nll)

checkpoint <- callback_model_checkpoint(
  paste0("checkpoints/GPD2.weights.h5"),
  monitor = "val_loss", verbose = 0,
  save_best_only = TRUE, save_weights_only = TRUE,
  mode = "min", save_freq = "epoch"
)

history <- GPD.model2 %>% fit(
  x = list("input_sigma" = X_train.excess,
           "input_xi" = as.matrix(X_train.excess[, 1])),
  y = as.matrix(Y_train.excess),
  batch_size = 16, epochs = 100, verbose = 0,
  callbacks = list(checkpoint, early.stopping),
  validation_split = 0.2
)
plot(history)
GPD.model2 <- GPD.model2 %>% load_model_weights(filepath = paste0("checkpoints/GPD2.weights.h5"))

test.GPD.estimates <- GPD.model2 %>% predict(list(X_test.excess, as.matrix(X_test.excess[, 1])))
test.sigma = test.GPD.estimates[, 1]
test.xi = test.GPD.estimates[, 2]
par(mfrow = c(1, 2))
hist(test.sigma, main = "sigma estimates")
hist(test.xi, main = "xi estimates")

In [ ]:
exp.data = apply(cbind(Y_test.excess, test.sigma, test.xi), 1, function(x) {
  qexp(evd::pgpd(x[1], loc = 0, scale = x[2], shape = x[3]))
})
n_p = length(exp.data)
ps = (1:n_p) / (n_p + 1)
qs = quantile(exp.data, ps, na.rm = T)
par(mfrow = c(1, 1))
plot(qexp(ps), qs, ylab = "Fitted", xlab = "Theoretical",
     main = "MLP model 2", pch = 20, asp = 1)
abline(a = 0, b = 1, col = "red")

test_loss_GPD2 <- GPD.model2 %>% evaluate(
  list(X_test.excess, as.matrix(X_test.excess[, 1])),
  as.matrix(Y_test.excess), batch_size = length(Y_test.excess))
print(paste0("test loss for GPD NN2: ", test_loss_GPD2))

**Model 3 (constant $\xi$).** Feed a vector of ones into a single bias-free neuron with exponential activation; its one weight $w$ gives $\xi = e^{w}$.

> **Comparison note.** `sigma_model` is reused here and is *not* redefined, so Model 3 starts from the $\sigma$-branch weights already trained inside Model 2. That warm start can favour Model 3 in the comparison below. Notebook **04** rebuilds every branch from scratch.

In [ ]:
xi_model_fixed <- input_xi %>%
  layer_dense(units = 1, use_bias = F, activation = 'exponential')

GPD.model3 <- keras_model(inputs = c(input_sigma, input_xi),
                          outputs = c(layer_concatenate(c(sigma_model, xi_model_fixed))))
GPD.model3 %>% compile(optimizer = "adam", loss = GPD_nll)

checkpoint <- callback_model_checkpoint(
  paste0("checkpoints/GPD3.weights.h5"),
  monitor = "val_loss", verbose = 0,
  save_best_only = TRUE, save_weights_only = TRUE,
  mode = "min", save_freq = "epoch"
)

history <- GPD.model3 %>% fit(
  x = list("input_sigma" = X_train.excess,
           "input_xi" = as.matrix(rep(1, nrow(X_train.excess)))),
  y = as.matrix(Y_train.excess),
  batch_size = 16, epochs = 100, verbose = 0,
  callbacks = list(checkpoint, early.stopping),
  validation_split = 0.2
)
plot(history)
GPD.model3 <- GPD.model3 %>% load_model_weights(filepath = paste0("checkpoints/GPD3.weights.h5"))

test.GPD.estimates <- GPD.model3 %>% predict(list(X_test.excess, as.matrix(rep(1, nrow(X_test.excess)))))
test.sigma = test.GPD.estimates[, 1]
test.xi = test.GPD.estimates[1, 2]
hist(test.sigma, main = "sigma estimates")
print(paste0("xi estimate: ", test.xi))

In [ ]:
exp.data = apply(cbind(Y_test.excess, test.sigma, test.xi), 1, function(x) {
  qexp(evd::pgpd(x[1], loc = 0, scale = x[2], shape = x[3]))
})
n_p = length(exp.data)
ps = (1:n_p) / (n_p + 1)
qs = quantile(exp.data, ps, na.rm = T)
plot(qexp(ps), qs, ylab = "Fitted", xlab = "Theoretical",
     main = "MLP model 3", pch = 20, asp = 1)
abline(a = 0, b = 1, col = "red")

Compare the models by test loss, plus a linear GPD model as a baseline.

> **Validation note.** Choosing the "best" model *on the test set* makes the winner's test loss slightly optimistic. Strictly, the choice should be made on validation data and the test set used once. Notebook **04** does this.
>
> **Note on the ranking.** The original text expects Model 3 to win. The ranking can change between runs and machines: these losses are *sums* over several hundred excesses, so differences of a few units are within training noise. Notebook **04** repeats the fit over several seeds for this reason.

In [ ]:
test_loss_GPD3 <- GPD.model3 %>% evaluate(
  list(X_test.excess, as.matrix(rep(1, nrow(X_test.excess)))),
  as.matrix(Y_test.excess), batch_size = length(Y_test.excess))

GPD.linear.model <- keras_model_sequential() %>%
  layer_dense(units = 2, activation = 'exponential')
GPD.linear.model %>% compile(optimizer = "adam", loss = GPD_nll)
history <- GPD.linear.model %>% fit(x = X_train.excess, y = as.matrix(Y_train.excess),
                                    batch_size = 16, epochs = 40, verbose = 0,
                                    validation_split = 0.2)
plot(history)
test_loss_linear_GPD <- GPD.linear.model %>% evaluate(X_test.excess, as.matrix(Y_test.excess),
                                                      batch_size = length(Y_test.excess))

print(paste0("test loss for GPD NN1: ", test_loss_GPD1))
print(paste0("test loss for GPD NN2: ", test_loss_GPD2))
print(paste0("test loss for GPD NN3 (homogeneous xi): ", test_loss_GPD3))
print(paste0("test loss for linear GPD: ", test_loss_linear_GPD))

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 5)
par(mfrow = c(1, 2))
t <- 3
X.plot <- X[t, , , ]
dim(X.plot) <- c(prod(dim(X.plot)[1:2]), dim(X.plot)[3])
X.plot.scaled <- apply(as.matrix(1:ncol(X.plot)), 1, function(ind)
  (X.plot[, ind] - means[ind]) / sds[ind])

image(log(Y[t, , ]), main = "Observation", asp = 0.625)
NN.predictions <- GPD.model3 %>% predict(list(X.plot.scaled, as.matrix(rep(1, nrow(X.plot.scaled)))))
sigma.predictions <- NN.predictions[, 1]
dim(sigma.predictions) <- dim(Y)[2:3]
sigma.predictions[Y[t, , ] < 0] = NA
image(log(sigma.predictions), main = "sigma prediction", asp = 0.625)
print(paste("GPD model 3, estimated xi: ", NN.predictions[1, 2]))

## 6 · Interpretation with LIME

**LIME** (Local Interpretable Model-agnostic Explanations) explains one prediction at a time. It perturbs a covariate vector $\mathbf x^*$, gets the network's prediction for each perturbed copy, and fits a weighted linear model in which copies closer to $\mathbf x^*$ get more weight. Here the prediction being explained is $\log \sigma(\mathbf x)$ from Model 3.

`lime` needs a `model_type` method and a `predict_model` method for the model's class.

**[Adapted]** The original defines these for the class name `keras.src.models.functional.Functional`. Here they are registered for whatever class name the installed Keras version reports, so the cell keeps working if that name changes.

The original feeds zeros instead of ones into the $\xi$ input. That is kept, and it is harmless: the $\sigma$ branch never sees that input.

In [ ]:
library(lime)
print(class(GPD.model3))

keras_cls <- class(GPD.model3)[1]
assign(paste0("model_type.", keras_cls), function(x, ...) "regression")
assign(paste0("predict_model.", keras_cls), function(x, newdata, type, ...) {
  pred <- x %>% predict(list(as.matrix(newdata), as.matrix(rep(0, nrow(newdata)))), verbose = 0)
  data.frame(log(pred[, 1]))
})

df.train = as.data.frame(X_train.excess)
names(df.train) <- cov.names
explainer <- lime::lime(x = df.train, model = GPD.model3, bin_continuous = F)

df.test = as.data.frame(X_test.excess)
names(df.test) <- cov.names

Explanations for the first 20 test covariate vectors, using the 10 most important features.

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 7)
explanation <- lime::explain(df.test[1:20, ], n_features = 10, explainer = explainer)
plot_explanations(explanation)

Explanations for the four largest test excesses.

In [ ]:
inds = order(Y_test.excess, decreasing = T)
explanation <- lime::explain(df.test[inds[1:4], ], n_features = 6, explainer = explainer)
plot_features(explanation)

## Summary

* A two-stage approach (deep quantile threshold, then deep GPD for excesses) gives covariate-dependent tail estimates.
* Simpler $\xi$ models often generalise as well as, or better than, fully flexible ones, because $\xi$ is weakly identified.
* Three evaluation choices here are reasonable for teaching but optimistic for real use: random row-level splits of spatially and temporally dependent data, scaling before splitting, and choosing the model on the test set. **Notebook 04** quantifies the first and fixes all three.